# 02 · Features

Construye las tablas que usarán el baseline (03) y el modelo (04).

**Parte 1 (este notebook, primera sección): puntos de K y D/ST.** nflverse trae `fantasy_points_ppr` para QB/RB/WR/TE, pero no para kickers ni defensas. Los calculo con `config/scoring.yaml` y **valido el cálculo contra los puntos reales de ESPN** en 2026.

**Parte 2: features** de forma reciente, uso, contexto del partido y rival para QB/RB/WR/TE, K y D/ST, calculadas solo con partidos anteriores y con una prueba automática de fuga de información.

> Solo temporada regular. Las credenciales se leen de `.env` y nunca se imprimen.

## 1. Setup

In [ ]:
import os
from pathlib import Path

import polars as pl
import yaml
import nflreadpy as nfl
from dotenv import load_dotenv
from espn_api.football import League
from espn_api.football.constant import PRO_TEAM_MAP

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
CONFIG = ROOT / "config"
DATA_PROC.mkdir(parents=True, exist_ok=True)
load_dotenv(ROOT / ".env")

SCORING = yaml.safe_load((CONFIG / "scoring.yaml").read_text(encoding="utf-8"))
VALIDATION = yaml.safe_load((CONFIG / "validation.yaml").read_text(encoding="utf-8"))

SEASONS = [2023, 2024, 2025, 2026]
REFRESH = False  # True para volver a descargar (2026 está en curso: refrescar cada semana)

pl.Config.set_tbl_rows(20)

### Esquema de validación (`config/validation.yaml`)

Walk-forward por temporada: cada fold entrena **solo** con temporadas anteriores a la que valida. La liga es nueva en 2026, así que solo 2026 tiene proyecciones de ESPN: en 2023–2025 el modelo se compara con un baseline simple (media móvil), y contra ESPN solo en 2026.

In [ ]:
pl.DataFrame([
    {"fold": f["name"], "entrena": ", ".join(map(str, f["train"])), "valida": str(f["valid"]), "compara contra": VALIDATION["baseline"]}
    for f in VALIDATION["folds"]
] + [{"fold": "holdout", "entrena": ", ".join(map(str, VALIDATION["holdout"]["train"])),
      "valida": str(VALIDATION["holdout"]["season"]), "compara contra": VALIDATION["holdout"]["compare_against"]}])

## 2. Datos

In [ ]:
def cached(name, loader):
    """Lee data/raw/<name>.parquet o lo descarga con `loader` si no existe (o si REFRESH)."""
    path = DATA_RAW / f"{name}.parquet"
    if path.exists() and not REFRESH:
        return pl.read_parquet(path)
    df = loader()
    df.write_parquet(path)
    return df

player_stats = cached("player_stats_weekly_2023_2026", lambda: nfl.load_player_stats(SEASONS, summary_level="week"))
team_stats = cached("team_stats_weekly_2023_2026", lambda: nfl.load_team_stats(SEASONS, summary_level="week"))
schedules = cached("schedules_2023_2026", lambda: nfl.load_schedules(SEASONS))

for name, df in [("player_stats", player_stats), ("team_stats", team_stats), ("schedules", schedules)]:
    print(f"{name:13s} {df.height:>7,} filas")

## 3. Puntos de kicker

| regla | columnas de nflverse |
|---|---|
| FG 0–39 (3) | `fg_made_0_19` + `fg_made_20_29` + `fg_made_30_39` |
| FG 40–49 (4), 50–59 (5), 60+ (6) | `fg_made_40_49`, `fg_made_50_59`, `fg_made_60_` |
| FG fallado (−1) | `fg_missed` **+ `fg_blocked`** |
| PAT (1) | `pat_made` |

ESPN cuenta un FG bloqueado como fallado; en nflverse `fg_missed` no incluye los bloqueados. Lo descubrí en la validación contra ESPN (sección 5).

In [ ]:
KICK = {r["abbr"]: r["points"] for r in SCORING["kicking"]}
FG_MADE = {"FG0": ["fg_made_0_19", "fg_made_20_29", "fg_made_30_39"],
           "FG40": ["fg_made_40_49"], "FG50": ["fg_made_50_59"], "FG60": ["fg_made_60_"]}
K_COLS = [c for cols in FG_MADE.values() for c in cols] + ["fg_att", "fg_made", "fg_missed", "fg_blocked", "pat_att", "pat_made"]

points_k = (player_stats
    .filter(pl.col("season_type") == "REG", pl.col("position") == "K")
    .with_columns(pl.col(K_COLS).fill_null(0))
    .with_columns(fantasy_points=(
        sum(pl.sum_horizontal(cols) * KICK[abbr] for abbr, cols in FG_MADE.items())
        + (pl.col("fg_missed") + pl.col("fg_blocked")) * KICK["FGM"]
        + pl.col("pat_made") * KICK["PAT"]))
    .select("season", "week", "game_id", "player_id", "player_display_name", "team", "opponent_team",
            "fg_att", "fg_made", "fg_missed", "fg_blocked", "pat_att", "pat_made", pl.col("fantasy_points").cast(pl.Float64)))

print(f"{points_k.height:,} partidos de kicker")
points_k.group_by("season").agg(pl.len().alias("partidos"), pl.col("fantasy_points").mean().round(2).alias("media"),
                                 pl.col("fantasy_points").max().alias("max")).sort("season")

## 4. Puntos de D/ST

**Eventos** (de `team_stats` del propio equipo): sacks, INTs, fumbles recuperados, safeties, bloqueos y TDs. En los TDs, nflverse separa `def_tds`, `fumble_recovery_tds` y `special_teams_tds`, así que sumo las tres columnas.

**Puntos y yardas permitidas:** las calculo con la definición de ESPN, que obtuve comparando con los valores reales de ESPN en la sección 5:
- **Puntos permitidos** = marcador del rival − 6 × (TDs de su defensa + TDs tras recuperar un fumble) − 2 × sus safeties. ESPN no le cuenta a tu D/ST los puntos que anota la defensa rival contra tu ofensiva.
- **Yardas permitidas** = yardas de pase del rival − yardas perdidas en sacks + yardas de carrera. Ojo: en nflverse `sack_yards_lost` viene con signo negativo, por eso uso el valor absoluto.

In [ ]:
DST = {r["abbr"]: r["points"] for r in SCORING["dst"]["events"]}

def bucket_points(col, rules):
    """Puntos según rangos inclusivos {min, max}; max=None = sin límite superior."""
    expr = pl.lit(None, dtype=pl.Float64)
    for r in reversed(rules):
        in_range = pl.col(col) >= r["min"] if r["max"] is None else pl.col(col).is_between(r["min"], r["max"])
        expr = pl.when(in_range).then(pl.lit(float(r["points"]))).otherwise(expr)
    return expr

TS_COLS = ["def_sacks", "def_interceptions", "fumble_recovery_opp", "def_safeties", "def_punt_blocks", "def_fg_blocks",
           "def_pat_blocks", "def_tds", "fumble_recovery_tds", "special_teams_tds", "def_2pt_made",
           "passing_yards", "sack_yards_lost", "rushing_yards"]
ts = (team_stats.filter(pl.col("season_type") == "REG")
      .select("season", "week", "team", *TS_COLS).with_columns(pl.col(TS_COLS).fill_null(0)))

# Un registro por equipo y partido jugado (sin byes ni partidos sin marcador)
games = schedules.filter(pl.col("game_type") == "REG", pl.col("home_score").is_not_null())
sides = pl.concat([
    games.select("season", "week", "game_id", team="home_team", opponent="away_team", opp_score="away_score"),
    games.select("season", "week", "game_id", team="away_team", opponent="home_team", opp_score="home_score"),
])
opp = ts.select("season", "week", opponent="team",
                opp_pass_yds="passing_yards", opp_sack_yds="sack_yards_lost", opp_rush_yds="rushing_yards",
                opp_def_tds="def_tds", opp_fr_tds="fumble_recovery_tds", opp_safeties="def_safeties")

points_dst = (sides
    .join(ts.drop("passing_yards", "sack_yards_lost", "rushing_yards"), on=["season", "week", "team"], how="left")
    .join(opp, on=["season", "week", "opponent"], how="left")
    .with_columns(
        points_allowed=pl.col("opp_score") - 6 * (pl.col("opp_def_tds") + pl.col("opp_fr_tds")) - 2 * pl.col("opp_safeties"),
        yards_allowed=pl.col("opp_pass_yds") - pl.col("opp_sack_yds").abs() + pl.col("opp_rush_yds"),
        tds=pl.col("def_tds") + pl.col("fumble_recovery_tds") + pl.col("special_teams_tds"),
        blocks=pl.col("def_punt_blocks") + pl.col("def_fg_blocks") + pl.col("def_pat_blocks"))
    .with_columns(
        pa_points=bucket_points("points_allowed", SCORING["dst"]["points_allowed"]),
        ya_points=bucket_points("yards_allowed", SCORING["dst"]["yards_allowed"]),
        event_points=(pl.col("def_sacks") * DST["SK"] + pl.col("def_interceptions") * DST["INT"]
                      + pl.col("fumble_recovery_opp") * DST["FR"] + pl.col("def_safeties") * DST["SF"]
                      + pl.col("blocks") * DST["BLKK"] + pl.col("tds") * DST["INTTD"]  # todos los TDs valen 6
                      + pl.col("def_2pt_made") * DST["2PTRET"]))
    .with_columns(fantasy_points=(pl.col("pa_points") + pl.col("ya_points") + pl.col("event_points")).cast(pl.Float64))
    .select("season", "week", "game_id", "team", "opponent", "points_allowed", "yards_allowed",
            "def_sacks", "def_interceptions", "fumble_recovery_opp", "def_safeties", "blocks", "tds",
            "pa_points", "ya_points", "event_points", "fantasy_points")
    .sort("season", "week", "team"))

n_null = points_dst["fantasy_points"].null_count()
print(f"{points_dst.height:,} partidos de D/ST · sin datos de team_stats: {n_null}")
assert n_null == 0, "Hay partidos sin team_stats: revisar joins"
points_dst.group_by("season").agg(pl.len().alias("partidos"), pl.col("fantasy_points").mean().round(2).alias("media"),
                                   pl.col("fantasy_points").min().alias("min"), pl.col("fantasy_points").max().alias("max")).sort("season")

## 5. Validación contra los puntos reales de ESPN (2026)

Para cada K y cada D/ST de la NFL descargo con `league.player_info` los puntos que ESPN les asignó en cada semana ya jugada de 2026 y los comparo con mi cálculo. Guardo también el detalle de ESPN (puntos y yardas permitidas, sacks) para poder explicar cada diferencia.

Así descubrí las tres reglas de las secciones 3 y 4 (FG bloqueado = fallado; ESPN excluye los TDs y safeties de la defensa rival de los puntos permitidos; signo de `sack_yards_lost`).

In [ ]:
league = League(league_id=int(os.environ["ESPN_LEAGUE_ID"]), year=2026,
                espn_s2=os.environ["ESPN_S2"], swid=os.environ["ESPN_SWID"])

# IDs de ESPN: los D/ST usan -16000 - proTeamId; los kickers se mapean con ff_playerids
ESPN_TO_NFLVERSE = {"WSH": "WAS", "LAR": "LA"}
dst_ids = pl.DataFrame([{"espn_id": -16000 - tid, "team": ESPN_TO_NFLVERSE.get(abbr, abbr)}
                        for tid, abbr in PRO_TEAM_MAP.items() if tid != 0])
ff_ids = nfl.load_ff_playerids().select("espn_id", "gsis_id").drop_nulls().unique("gsis_id")

k26 = points_k.filter(pl.col("season") == 2026).join(ff_ids, left_on="player_id", right_on="gsis_id", how="left")
WEEKS = sorted(points_dst.filter(pl.col("season") == 2026)["week"].unique().to_list())
print(f"Semanas 2026 jugadas: {WEEKS} · K sin espn_id: {k26['espn_id'].null_count()}")

In [ ]:
actuals_path = DATA_RAW / "espn_actuals_k_dst_2026.parquet"
need_fetch = REFRESH or not actuals_path.exists() or \
             sorted(pl.read_parquet(actuals_path)["week"].unique().to_list()) != WEEKS

if need_fetch:
    ids = k26["espn_id"].drop_nulls().unique().to_list() + dst_ids["espn_id"].to_list()
    rows = []
    for i in range(0, len(ids), 25):
        res = league.player_info(playerId=ids[i:i + 25])
        for p in res if isinstance(res, list) else [res]:
            for wk in WEEKS:
                wk_stats = p.stats.get(wk, {})
                if "points" not in wk_stats:
                    continue
                raw = wk_stats.get("breakdown", {})
                rows.append({"espn_id": p.playerId, "name": p.name, "pos": p.position, "week": wk,
                             "espn_points": float(wk_stats["points"]),
                             "espn_pa": raw.get("defensivePointsAllowed"), "espn_ya": raw.get("defensiveYardsAllowed"),
                             "espn_sacks": raw.get("defensiveSacks")})
    actuals = pl.DataFrame(rows, schema_overrides={"espn_pa": pl.Float64, "espn_ya": pl.Float64, "espn_sacks": pl.Float64})
    actuals.write_parquet(actuals_path)
else:
    actuals = pl.read_parquet(actuals_path)
actuals.group_by("pos").agg(pl.len().alias("partidos"), pl.col("espn_id").n_unique().alias("jugadores"))

In [ ]:
TOL = 1e-6
MIN_MATCH = 0.95  # por debajo de esto la fórmula está mal, no son simples correcciones de datos

val_k = (actuals.filter(pl.col("pos") == "K")
         .join(k26.select("espn_id", "week", "player_display_name", "fg_missed", "fg_blocked", "fantasy_points"),
               on=["espn_id", "week"], how="left")
         .with_columns(diff=pl.col("espn_points") - pl.col("fantasy_points")))
val_dst = (actuals.filter(pl.col("pos") == "D/ST").join(dst_ids, on="espn_id")
           .join(points_dst.filter(pl.col("season") == 2026), on=["team", "week"], how="left")
           .with_columns(diff=pl.col("espn_points") - pl.col("fantasy_points")))

summary = []
for name, v in [("K", val_k), ("D/ST", val_dst)]:
    ok = (v["diff"].abs() < TOL).sum()
    summary.append({"posición": name, "partidos": v.height, "coinciden": ok, "pct": round(ok / v.height * 100, 1),
                    "sin cálculo": v["fantasy_points"].null_count()})
summary.append({"posición": "D/ST: puntos permitidos", "partidos": val_dst.height,
                "coinciden": (val_dst["points_allowed"] == val_dst["espn_pa"]).sum(), "pct": None, "sin cálculo": None})
summary.append({"posición": "D/ST: yardas permitidas", "partidos": val_dst.height,
                "coinciden": (val_dst["yards_allowed"] == val_dst["espn_ya"]).sum(), "pct": None, "sin cálculo": None})
summary = pl.DataFrame(summary)

for row in summary.filter(pl.col("pct").is_not_null()).iter_rows(named=True):
    if row["pct"] / 100 < MIN_MATCH:
        raise ValueError(f"{row['posición']}: solo {row['pct']}% coincide con ESPN: revisar la fórmula")
summary

Diferencias restantes y su causa probable:

In [ ]:
(pl.concat([
    val_k.filter(pl.col("diff").abs() > TOL)
         .select(pl.col("name"), "week", "espn_points", "fantasy_points", "diff",
                 detalle=pl.format("fg_missed={} fg_blocked={}", "fg_missed", "fg_blocked")),
    val_dst.filter(pl.col("diff").abs() > TOL)
           .select(pl.col("name"), "week", "espn_points", "fantasy_points", "diff",
                   detalle=pl.format("sacks nflverse={} / ESPN={} · PA {} / {} · YA {} / {}",
                                     "def_sacks", "espn_sacks", "points_allowed", "espn_pa", "yards_allowed", "espn_ya")),
], how="vertical_relaxed"))

Si solo queda alguna diferencia puntual en una estadística (por ejemplo, un sack de más en ESPN) y los puntos y yardas permitidas cuadran, es una discrepancia entre las fuentes de datos o una corrección de estadísticas posterior, no un error de la fórmula.

## 6. Guardar

In [ ]:
points_dst = points_dst.join(dst_ids, on="team", how="left")  # espn_id para cruzar con rosters de ESPN
points_k = points_k.join(ff_ids.rename({"gsis_id": "player_id"}), on="player_id", how="left")

points_k.write_parquet(DATA_PROC / "points_k.parquet")
points_dst.write_parquet(DATA_PROC / "points_dst.parquet")
for p in ["points_k.parquet", "points_dst.parquet"]:
    print(f"✓ data/processed/{p}")

## 7. Tabla base de QB/RB/WR/TE

Una fila por jugador y partido de temporada regular.

- **Partidos sin estadísticas:** `player_stats` solo tiene filas de jugadores que registraron alguna estadística. Un TE que jugó 30 snaps bloqueando no aparece, y eso inflaría los promedios. Por eso uno la tabla con `snap_counts` y agrego esos partidos con 0 en todas las estadísticas.
- **Puntos esperados (`xfp`):** vienen de `ff_opportunity` y son los puntos que suele producir ese volumen de uso (targets, carries, distancia al end zone), sin la suerte de los TDs.

In [ ]:
POS = ["QB", "RB", "WR", "TE"]
STAT_COLS = ["attempts", "completions", "passing_yards", "passing_tds", "passing_interceptions",
             "carries", "rushing_yards", "rushing_tds",
             "targets", "receptions", "receiving_yards", "receiving_tds", "receiving_air_yards",
             "target_share", "air_yards_share", "wopr", "fantasy_points_ppr"]
KEYS = ["season", "week"]
as_int = [pl.col("season").cast(pl.Int32), pl.col("week").cast(pl.Int32)]

playerids = cached("ff_playerids", nfl.load_ff_playerids)
snap_counts = cached("snap_counts_2023_2026", lambda: nfl.load_snap_counts(SEASONS))
opportunity = cached("ff_opportunity_2023_2026", lambda: nfl.load_ff_opportunity(SEASONS))

stats_off = (player_stats
    .filter(pl.col("season_type") == "REG", pl.col("position").is_in(POS))
    .select(*as_int, "player_id", "player_display_name", "position", "team", "opponent_team", *STAT_COLS))

pfr_to_gsis = playerids.select("pfr_id", "gsis_id").drop_nulls().unique("pfr_id").unique("gsis_id")
snaps = (snap_counts
    .filter(pl.col("game_type") == "REG", pl.col("position").is_in(POS), pl.col("offense_snaps") > 0)
    .join(pfr_to_gsis, left_on="pfr_player_id", right_on="pfr_id")
    .select(*as_int, player_id="gsis_id", snap_name="player", snap_position="position",
            snap_team="team", snap_opponent="opponent", offense_pct="offense_pct"))

xfp = (opportunity
    .filter(pl.col("player_id").is_not_null())
    .select(*as_int, "player_id", xfp="total_fantasy_points_exp")
    .group_by(*KEYS, "player_id").agg(pl.col("xfp").sum()))

base = (stats_off
    .join(snaps, on=[*KEYS, "player_id"], how="full", coalesce=True)
    .with_columns(
        from_snaps_only=pl.col("player_display_name").is_null(),
        player_display_name=pl.coalesce("player_display_name", "snap_name"),
        position=pl.coalesce("position", "snap_position"),
        team=pl.coalesce("team", "snap_team"),
        opponent_team=pl.coalesce("opponent_team", "snap_opponent"))
    .drop("snap_name", "snap_position", "snap_team", "snap_opponent")
    .with_columns(pl.col(STAT_COLS).fill_null(0))
    .join(xfp, on=[*KEYS, "player_id"], how="left")
    # sin ninguna oportunidad (0 pases, carries y targets) los puntos esperados son 0
    .with_columns(xfp=pl.when(pl.col("xfp").is_null() & (pl.col("attempts") + pl.col("carries") + pl.col("targets") == 0))
                        .then(0.0).otherwise(pl.col("xfp")))
    .sort("player_id", *KEYS))

assert base.select("player_id", *KEYS).is_duplicated().sum() == 0, "Filas duplicadas jugador-semana"
print(f"{base.height:,} partidos · agregados desde snap_counts con 0 puntos: {base['from_snaps_only'].sum():,} · "
      f"sin xfp: {base['xfp'].null_count():,} · sin % de snaps: {base['offense_pct'].null_count():,}")
base.group_by("position").agg(pl.len().alias("partidos"), pl.col("fantasy_points_ppr").mean().round(2).alias("ppr_media"),
                              pl.col("from_snaps_only").mean().round(3).alias("pct_solo_snaps")).sort("position")

## 8. Contexto del partido (`schedules`)

Una fila por equipo y partido, **incluidos los que aún no se juegan**, para poder predecir la semana siguiente en el notebook 05.

- **Líneas de apuestas:** `spread_line` es positivo cuando el local es favorito (su correlación con el resultado es positiva). Con él y el total calculo los puntos que se espera que anote cada equipo: `implied_team = (total + spread) / 2`, desde la perspectiva de cada equipo.
- **Estadio techado (`indoor`):** domo o techo cerrado. En esos partidos pongo viento = 0 y temperatura = 70 °F, porque nflverse los deja vacíos.
- **Clima faltante:** nflverse no trae el clima de unos 50 partidos jugados al aire libre (sobre todo de 2023). Los dejo como nulos; los modelos de árboles los manejan bien.
- ⚠ Las líneas son **de cierre** (ver limitaciones).

In [ ]:
sched = schedules.filter(pl.col("game_type") == "REG")

def side(home: bool):
    t, o, sign = ("home", "away", 1) if home else ("away", "home", -1)
    return sched.select(*as_int, "game_id",
                        team=pl.col(f"{t}_team"), opponent=pl.col(f"{o}_team"),
                        team_score=pl.col(f"{t}_score"), opp_score=pl.col(f"{o}_score"),
                        is_home=pl.lit(int(home), dtype=pl.Int8), rest_days=pl.col(f"{t}_rest"),
                        spread=sign * pl.col("spread_line"),  # > 0: el equipo es favorito
                        total_line=pl.col("total_line"), roof=pl.col("roof"),
                        temp=pl.col("temp").cast(pl.Float64), wind=pl.col("wind").cast(pl.Float64),
                        div_game=pl.col("div_game").cast(pl.Int8))

team_games = pl.concat([side(True), side(False)])
context = (team_games
    .with_columns(implied_team=(pl.col("total_line") + pl.col("spread")) / 2,
                  implied_opp=(pl.col("total_line") - pl.col("spread")) / 2,
                  indoor=pl.col("roof").is_in(["dome", "closed"]).cast(pl.Int8))
    .with_columns(wind=pl.when(pl.col("indoor") == 1).then(0.0).otherwise(pl.col("wind")),
                  temp=pl.when(pl.col("indoor") == 1).then(70.0).otherwise(pl.col("temp")))
    .select(*KEYS, "team", "is_home", "rest_days", "spread", "total_line", "implied_team", "implied_opp",
            "indoor", "temp", "wind", "div_game"))
CONTEXT_FEATS = [c for c in context.columns if c not in (*KEYS, "team")]

played = context.join(team_games.filter(pl.col("team_score").is_not_null()).select(*KEYS, "team"), on=[*KEYS, "team"])
print(f"{context.height:,} filas equipo-partido ({played.height:,} ya jugadas)")
print(f"  jugadas sin línea de apuestas: {played['spread'].null_count()} · jugadas sin clima: {played['wind'].null_count()}")
print(f"  futuras sin línea todavía: {context['spread'].null_count() - played['spread'].null_count()}")

## 9. Features sin fuga de información

Todas las features de forma reciente se calculan con `shift(1)` dentro de cada jugador o equipo: la fila de la semana *w* solo ve partidos anteriores.

| sufijo | significado |
|---|---|
| `_l3`, `_l5` | media de los últimos 3 / 5 partidos jugados (cruza temporadas) |
| `_std` | media de la temporada hasta la semana anterior (*season to date*) |
| `_prev` | media de toda la temporada anterior |
| `games_season`, `games_career` | partidos previos en la temporada / en total (0 = debut) |
| `weeks_since_last` | semanas desde su último partido en la temporada (>1 = bye o ausencia) |

Las medias `_l3` y `_l5` cruzan temporadas a propósito: en la semana 1 usan el final de la temporada anterior. `games_season` le dice al modelo cuánto de eso es del año actual.

**Rival:** para cada defensa y posición sumo los puntos PPR que permitió por partido y calculo su media de los últimos 5 partidos y de la temporada (por ejemplo, cuántos puntos suele permitir a los WR).

In [ ]:
def add_rolling(df, key, cols, windows=(3, 5), prev_season=True):
    """Añade medias de partidos ANTERIORES de cada `key` (nunca incluye la fila actual)."""
    key = [key] if isinstance(key, str) else list(key)
    df = df.sort(*key, *KEYS)
    exprs = []
    for c in cols:
        prev = pl.col(c).shift(1)
        exprs += [prev.rolling_mean(w, min_samples=1).over(key).alias(f"{c}_l{w}") for w in windows]
        exprs.append((prev.cum_sum() / prev.cum_count()).over([*key, "season"]).alias(f"{c}_std"))
    df = df.with_columns(exprs)
    if prev_season:
        prev = (df.group_by(*key, "season").agg([pl.col(c).mean().alias(f"{c}_prev") for c in cols])
                  .with_columns(pl.col("season") + 1))
        df = df.join(prev, on=[*key, "season"], how="left")
    return df

def add_counts(df, key):
    same_season = pl.col("season") == pl.col("season").shift(1).over(key)
    return df.with_columns(
        games_season=pl.int_range(pl.len()).over([key, "season"]),
        games_career=pl.int_range(pl.len()).over(key),
        weeks_since_last=pl.when(same_season).then(pl.col("week") - pl.col("week").shift(1).over(key)))

def feature_cols(df, exclude):
    return [c for c in df.columns if c not in exclude]

### QB/RB/WR/TE

In [ ]:
OFF_ROLL = ["fantasy_points_ppr", "xfp", "offense_pct", "attempts", "passing_yards", "passing_tds",
            "carries", "rushing_yards", "targets", "receptions", "receiving_yards", "receiving_air_yards",
            "target_share", "air_yards_share", "wopr"]
OFF_IDS = ["season", "week", "player_id", "player_display_name", "position", "team", "opponent_team", "from_snaps_only"]

def build_offense(base):
    f = add_counts(add_rolling(base, "player_id", OFF_ROLL), "player_id")

    allowed = (base.group_by(*KEYS, "opponent_team", "position")
                   .agg(pl.col("fantasy_points_ppr").sum().alias("opp_ppr_allowed"))
                   .rename({"opponent_team": "defense"}))
    allowed = add_rolling(allowed, ["defense", "position"], ["opp_ppr_allowed"], windows=(5,), prev_season=False)

    f = (f.join(allowed.drop("opp_ppr_allowed"), left_on=[*KEYS, "opponent_team", "position"],
                right_on=[*KEYS, "defense", "position"], how="left")
          .join(context, on=[*KEYS, "team"], how="left")
          .with_columns(y=pl.col("fantasy_points_ppr")))
    feats = [c for c in f.columns if c.endswith(("_l3", "_l5", "_std", "_prev"))] \
            + ["games_season", "games_career", "weeks_since_last"] + CONTEXT_FEATS
    return f.select(*OFF_IDS, "y", *feats).sort("player_id", *KEYS)

features_offense = build_offense(base)
OFF_FEATS = feature_cols(features_offense, [*OFF_IDS, "y"])
print(f"{features_offense.height:,} filas · {len(OFF_FEATS)} features")

### K y D/ST

In [ ]:
K_IDS = ["season", "week", "player_id", "player_display_name", "team", "opponent_team", "espn_id"]

def build_k(points_k):
    f = add_counts(add_rolling(points_k, "player_id", ["fantasy_points", "fg_att", "fg_made", "pat_att"]), "player_id")
    f = f.join(context, on=[*KEYS, "team"], how="left").with_columns(y=pl.col("fantasy_points"))
    feats = [c for c in f.columns if c.endswith(("_l3", "_l5", "_std", "_prev"))] \
            + ["games_season", "games_career", "weeks_since_last"] + CONTEXT_FEATS
    return f.select(*K_IDS, "y", *feats).sort("player_id", *KEYS)

# Ofensiva de cada equipo por partido: lo que enfrenta un D/ST
team_offense = (team_games.filter(pl.col("team_score").is_not_null())
    .select(*KEYS, "team", points_scored="team_score")
    .join(ts.select("season", "week", "team").join(
              team_stats.filter(pl.col("season_type") == "REG").select(
                  "season", "week", "team", sacks_allowed="sacks_suffered",
                  giveaways=pl.col("passing_interceptions").fill_null(0) + pl.col("sack_fumbles_lost").fill_null(0)
                            + pl.col("rushing_fumbles_lost").fill_null(0) + pl.col("receiving_fumbles_lost").fill_null(0)),
              on=["season", "week", "team"]).with_columns(*as_int),
          on=[*KEYS, "team"], how="left"))

DST_IDS = ["season", "week", "team", "opponent", "espn_id"]

def build_dst(points_dst, team_offense):
    own = points_dst.with_columns(takeaways=pl.col("def_interceptions") + pl.col("fumble_recovery_opp"))
    f = add_counts(add_rolling(own, "team", ["fantasy_points", "def_sacks", "takeaways", "points_allowed", "yards_allowed"]), "team")
    opp = add_rolling(team_offense, "team", ["points_scored", "sacks_allowed", "giveaways"], windows=(5,), prev_season=False)
    opp_feats = [c for c in opp.columns if c.endswith(("_l5", "_std"))]
    opp = opp.select(*KEYS, pl.col("team").alias("opponent"), *[pl.col(c).alias(f"opp_{c}") for c in opp_feats])
    f = (f.join(opp, on=[*KEYS, "opponent"], how="left")
          .join(context, on=[*KEYS, "team"], how="left")
          .with_columns(y=pl.col("fantasy_points")))
    feats = [c for c in f.columns if c.endswith(("_l3", "_l5", "_std", "_prev"))] \
            + ["games_season", "games_career", "weeks_since_last"] + CONTEXT_FEATS
    return f.select(*DST_IDS, "y", *feats).sort("team", *KEYS)

points_k = points_k.with_columns(*as_int)
points_dst = points_dst.with_columns(*as_int)
features_k = build_k(points_k)
features_dst = build_dst(points_dst, team_offense)
K_FEATS = feature_cols(features_k, [*K_IDS, "y"])
DST_FEATS = feature_cols(features_dst, [*DST_IDS, "y"])
print(f"K:    {features_k.height:,} filas · {len(K_FEATS)} features")
print(f"D/ST: {features_dst.height:,} filas · {len(DST_FEATS)} features")

## 10. Pruebas

### Fuga de información
Multiplico por 100 **todas** las estadísticas de una semana (2025, semana 10) y reconstruyo las features. Si no hay fuga:
- las features de esa semana y de las anteriores no cambian (no pueden ver el dato alterado);
- las de semanas posteriores **sí** cambian, lo que confirma que la prueba detectaría una fuga si la hubiera.

In [ ]:
from polars.testing import assert_frame_equal

LEAK_SEASON, LEAK_WEEK = 2025, 10
is_leak_week = (pl.col("season") == LEAK_SEASON) & (pl.col("week") == LEAK_WEEK)
up_to = (pl.col("season") < LEAK_SEASON) | ((pl.col("season") == LEAK_SEASON) & (pl.col("week") <= LEAK_WEEK))

def corrupt(df, cols):
    return df.with_columns([pl.when(is_leak_week).then(pl.col(c) * 100 + 7).otherwise(pl.col(c)).alias(c) for c in cols])

def leakage_test(name, original, rebuilt, keys, feats):
    a, b = original.sort(keys), rebuilt.sort(keys)
    assert_frame_equal(a.filter(up_to).select(feats), b.filter(up_to).select(feats))
    changed = not a.filter(~up_to).select(feats).equals(b.filter(~up_to).select(feats))
    assert changed, f"{name}: la alteración no afectó semanas posteriores; la prueba no es sensible"
    print(f"✓ {name}: sin fuga (features hasta {LEAK_SEASON} sem {LEAK_WEEK} idénticas; posteriores cambian)")

num = lambda df, exclude: [c for c, t in df.schema.items() if t.is_numeric() and c not in exclude]

leakage_test("QB/RB/WR/TE", features_offense,
             build_offense(corrupt(base, num(base, KEYS))), ["player_id", *KEYS], OFF_FEATS)
leakage_test("K", features_k,
             build_k(corrupt(points_k, num(points_k, [*KEYS, "espn_id"]))), ["player_id", *KEYS], K_FEATS)
leakage_test("D/ST", features_dst,
             build_dst(corrupt(points_dst, num(points_dst, [*KEYS, "espn_id"])),
                       corrupt(team_offense, num(team_offense, KEYS))), ["team", *KEYS], DST_FEATS)

### Duplicados y nulos

In [ ]:
for name, df, keys in [("QB/RB/WR/TE", features_offense, ["player_id", *KEYS]),
                       ("K", features_k, ["player_id", *KEYS]), ("D/ST", features_dst, ["team", *KEYS])]:
    assert df.select(keys).is_duplicated().sum() == 0, f"{name}: filas duplicadas"
    assert df["y"].null_count() == 0, f"{name}: objetivo nulo"
print("✓ Sin duplicados ni objetivos nulos")

# Nulos esperados: primeras apariciones (sin partidos previos) y partidos sin línea/clima
def null_report(df, feats):
    return (df.select(pl.col(feats).null_count()).transpose(include_header=True, header_name="feature", column_names=["nulos"])
              .with_columns((pl.col("nulos") / df.height * 100).round(1).alias("pct")).sort("nulos", descending=True))

null_report(features_offense, OFF_FEATS).head(10)

Los `_prev` son los que más nulos tienen, lo cual es esperable: son todas las filas de 2023 (no hay 2022 en los datos) más los novatos. Los `_std` son nulos en el primer partido de cada temporada.

### ¿Las features tienen sentido?
Correlación de algunas features con el objetivo, por posición. Es solo una comprobación de sentido común, no una selección de features.

In [ ]:
CHECK = ["fantasy_points_ppr_l5", "xfp_l5", "fantasy_points_ppr_std", "offense_pct_l3", "implied_team", "opp_ppr_allowed_l5"]
(features_offense.filter(pl.col("games_career") >= 3)
 .group_by("position")
 .agg([pl.corr(c, "y").round(3).alias(c) for c in CHECK])
 .sort("position"))

## 11. Guardar

In [ ]:
for name, df in [("features_offense", features_offense), ("features_k", features_k), ("features_dst", features_dst)]:
    df.write_parquet(DATA_PROC / f"{name}.parquet")
    print(f"✓ data/processed/{name}.parquet · {df.height:,} filas × {df.width} columnas")

Cada tabla tiene **identificadores**, el **objetivo `y`** (puntos del partido) y **features**, pero no las estadísticas del propio partido. Así el notebook 04 no puede usarlas por error. Las features son todas las columnas que no son identificadores ni `y`.

## Supuestos y limitaciones

- **Las líneas de apuestas son de cierre.** `schedules` de nflverse trae el spread y el total *de cierre* (justo antes del partido). Si la alineación se decide días antes, solo estarán las líneas de apertura o las del momento, así que el backtest es algo optimista respecto al uso real.
- **El modelo supone que el jugador juega.** El objetivo solo existe para partidos jugados: el modelo predice los puntos *si juega*, no la probabilidad de que juegue. Lesiones, inactivos y descansos se manejan fuera del modelo (con el estado de lesión de ESPN al decidir la alineación).
- **Validación de K y D/ST limitada a 2026.** La fórmula se verificó contra ESPN solo en las semanas jugadas de 2026, porque la liga es nueva. Para 2023–2025 se asume que las reglas de ESPN no cambiaron.
- **Casos raros de D/ST sin verificar:** si ESPN resta de los puntos permitidos los TDs de retorno (kickoff o punt) del rival, y cómo registra nflverse los retornos de 2 pts (`def_2pt_made`). No aparecieron en la muestra de validación. La safety de 1 punto no se modela.
- **Correcciones de estadísticas.** ESPN y nflverse pueden diferir en algún dato puntual (por ejemplo, sacks compartidos o corregidos después del partido).